# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. All references to dataset elements (record sets, fields, columns) use their `@id` as required by the Croissant specification.

### Dataset Source
The dataset source is the Croissant schema hosted at:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and print the metadata
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs as specified in the Croissant schema. This step provides context on what tabular data is available for extraction and analysis.

In [ ]:
# List all available record sets and their fields by @id
record_sets = metadata.record_set
print("Available Record Sets and their field @ids:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    for field in rs.field:
        print(f"    - Field @id: {field.id} | name: {field.name}")

## 3. Data Extraction
Load a record set of interest into a pandas DataFrame. Below, we select a primary tabular record set by its `@id`, extract its records, and examine the available fields and sample records.

**Note:** Update the `selected_record_set_id` variable to reference the actual record set `@id` you wish to explore. All field/column accesses must use their `@id`.

In [ ]:
# Choose the record set to extract by its @id (replace with your record set's @id as needed)
selected_record_set = record_sets[0]  # For this dataset, assuming the first RecordSet contains the main table
selected_record_set_id = selected_record_set.id

# Extract the records using the record set @id
records = list(dataset.records(record_set=selected_record_set_id))
df = pd.DataFrame(records)
print(f"Loaded DataFrame for RecordSet '{selected_record_set_id}' with columns:")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Here we demonstrate common processing tasks on the data for one record set field:
- Filtering records based on field values
- Normalizing a numeric field
- Grouping by a categorical field

**All field references use their `@id`**.

In [ ]:
# Select numeric and group fields by their @id
# For demonstration, try to use the first numeric field and first non-numeric as group
import numpy as np

numeric_field_id = None
group_field_id = None
for field in selected_record_set.field:
    # Try to detect the first numeric field (by dataType)
    if hasattr(field, 'data_type'):
        if field.data_type in ('Integer', 'Float', 'Number') and field.id in df.columns:
            numeric_field_id = field.id
            continue
    # Try to detect a likely group field (text/category)
    if not group_field_id and hasattr(field, 'data_type') and field.data_type in ('Text', ) and field.id in df.columns:
        group_field_id = field.id

if numeric_field_id is None:
    print("No numeric field found for EDA.")
else:
    # Handle missing/non-numeric as needed
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records in RecordSet '{selected_record_set_id}' where {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_id, as_index=False)
            .agg({numeric_field_id: 'mean', f"{numeric_field_id}_normalized": 'mean'})
        )
        print(f"\nGrouped data by '{group_field_id}':")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships using the extracted DataFrame. Below is an example histogram for the selected numeric field and a boxplot by group (if grouping field is found).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}'")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"'{numeric_field_id}' by '{group_field_id}'")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- The FAIR² Croissant dataset was loaded and inspected using `mlcroissant`, referencing all schema elements by `@id`.
- A main record set was extracted, its fields reviewed, and basic filtering, normalization, and grouping were performed on a numeric field.
- Visualizations illustrate field distributions and (if applicable) differences by categorical field.

_To extend this notebook, select different record sets or fields by their `@id`, or implement additional analyses as required for your domain._